In [1]:
import os
os.chdir("..")
import sys
sys.path.insert(0, "./src")

import json
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.train import train_catboost

In [2]:
def preprocess_autoscout24(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop rows with missing or invalid prices
    df["price_in_euro"] = pd.to_numeric(df["price_in_euro"], errors="coerce")
    df = df[df["price_in_euro"].notna() & (df["price_in_euro"] > 500)]

    # Numerical
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df.loc[(df["year"] < 1950) | (df["year"] > 2026), "year"] = np.nan

    df["power_kw"] = pd.to_numeric(df["power_kw"], errors="coerce")
    df["power_ps"] = pd.to_numeric(df["power_ps"], errors="coerce")
    df["mileage_in_km"] = pd.to_numeric(df["mileage_in_km"], errors="coerce")

    # Categorical: lowercase and strip
    for col in ["brand", "model", "color", "transmission_type", "fuel_type"]:
        df[col] = df[col].str.lower().str.strip().fillna("unknown")

    # Target
    df["price"] = df["price_in_euro"]

    keep = [
        "brand", "model", "color", "transmission_type", "fuel_type",
        "year", "power_kw", "power_ps", "mileage_in_km",
        "price",
    ]
    return df[keep].reset_index(drop=True)


df_raw = pd.read_csv("./data/autoscout24.csv")
df_clean = preprocess_autoscout24(df_raw)
print(df_clean.shape)
df_clean.head(3)

(250723, 10)


,brand,model,color,transmission_type,fuel_type,year,power_kw,power_ps,mileage_in_km,price
0,alfa-romeo,alfa romeo gtv,red,manual,petrol,1995.0,148.0,201.0,160500.0,1300.0
1,alfa-romeo,alfa romeo 164,black,manual,petrol,1995.0,191.0,260.0,190000.0,24900.0
2,alfa-romeo,alfa romeo spider,black,unknown,petrol,1995.0,110.0,150.0,129000.0,5900.0


In [3]:
_NULLISH = {"nan", "none", "other", "unknown", "n/a", "na", "", "unspecified", "not specified", "missing"}
_DESC_COLS = ["brand", "model", "color", "year", "power_kw", "power_ps", "transmission_type", "fuel_type"]

def make_desc(row: pd.Series) -> str:
    """Recreate the description key used during normalization."""
    parts = []
    for col in _DESC_COLS:
        v = row.get(col)
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s.lower() in _NULLISH:
            continue
        # year and power_kw are floats in CSV — format without .0 suffix
        if isinstance(v, float) and v == int(v):
            s = str(int(v))
        parts.append(f"{col}: {s}")
    return ", ".join(parts)


# Build descriptions from the original (non-lowercased) CSV so model/color case matches DB keys
_price_numeric = pd.to_numeric(df_raw["price_in_euro"], errors="coerce")
df_orig = df_raw[_price_numeric.notna() & (_price_numeric > 500)].reset_index(drop=True).copy()

df_clean["_desc"] = df_orig.apply(make_desc, axis=1)

# Load mappings from SQLite
conn = sqlite3.connect("./data/autoscout24_mappings.db")
cursor = conn.execute("SELECT description, result FROM mappings")
mappings = {row[0]: json.loads(row[1]) for row in cursor}
conn.close()
print(f"Loaded {len(mappings):,} mappings")

# Expand all normalized fields as n_ columns
mapping_keys = {k for v in mappings.values() for k in v if k != "equipment"}

for key in sorted(mapping_keys):
    df_clean[f"n_{key}"] = df_clean["_desc"].map(
        lambda d, k=key: (mappings.get(d) or {}).get(k)
    )

n_cols = [c for c in df_clean.columns if c.startswith("n_")]
print(f"Added {len(n_cols)} n_ columns: {n_cols}")

# Coverage check
matched = df_clean["_desc"].isin(mappings)
print(f"Coverage: {matched.sum():,} / {len(df_clean):,} rows ({matched.mean()*100:.1f}%)")

Loaded 96,294 mappings
Added 27 n_ columns: ['n_body_type', 'n_brand', 'n_color', 'n_condition', 'n_cylinders', 'n_doors', 'n_drive_type', 'n_energy_source', 'n_engine_aspiration', 'n_engine_power', 'n_engine_power_unit', 'n_engine_size', 'n_engine_size_unit', 'n_fuel_consumption', 'n_fuel_consumption_unit', 'n_fuel_type', 'n_injection_type', 'n_mileage', 'n_mileage_unit', 'n_model', 'n_propulsion_system', 'n_submodel', 'n_transmission_gears', 'n_transmission_technology', 'n_transmission_type', 'n_trim_level', 'n_year']
Coverage: 250,713 / 250,723 rows (100.0%)


In [4]:
df_clean.sample(10)

,brand,model,color,transmission_type,fuel_type,year,power_kw,power_ps,mileage_in_km,price,...,n_mileage,n_mileage_unit,n_model,n_propulsion_system,n_submodel,n_transmission_gears,n_transmission_technology,n_transmission_type,n_trim_level,n_year
33291,bmw,bmw 430,grey,automatic,diesel,2015.0,190.0,258.0,131500.0,25990.0,...,None,None,430,None,None,None,None,automatic,None,2015.0
61578,ford,ford focus,blue,manual,diesel,2014.0,85.0,116.0,144895.0,8480.0,...,None,None,focus,None,None,None,None,manual,None,2014.0
16896,audi,audi a6,grey,automatic,petrol,2019.0,250.0,340.0,19306.0,44590.0,...,None,None,a6,None,None,None,None,automatic,None,2019.0
161452,peugeot,peugeot 2008,grey,automatic,petrol,2019.0,81.0,110.0,28700.0,17200.0,...,None,None,2008,None,None,None,None,automatic,None,2019.0
197322,skoda,skoda superb,blue,automatic,hybrid,2020.0,160.0,218.0,56200.0,23990.0,...,None,None,superb,None,None,None,None,automatic,None,2020.0
210693,toyota,toyota yaris,red,automatic,hybrid,2022.0,68.0,92.0,17750.0,21180.0,...,None,None,yaris,None,None,None,None,automatic,None,2022.0
30631,bmw,bmw 120,silver,manual,diesel,2012.0,135.0,184.0,189652.0,17499.0,...,None,None,120,None,None,None,None,manual,None,2012.0
143363,opel,opel adam,black,manual,petrol,2013.0,64.0,87.0,80600.0,8990.0,...,None,None,adam,None,None,None,None,manual,None,2013.0
86038,jeep,jeep renegade,beige,automatic,diesel,2017.0,125.0,170.0,106500.0,22990.0,...,None,None,renegade,None,None,None,None,automatic,None,2017.0
246391,volkswagen,volkswagen grand california,white,automatic,diesel,2023.0,130.0,177.0,10.0,73880.0,...,None,None,grand california,None,None,None,None,automatic,None,2023.0


In [5]:
# --- Raw (non-normalized) columns ---
RAW_NUM = ["year", "power_kw", "power_ps", "mileage_in_km"]
RAW_CAT = ["brand", "model", "color", "transmission_type", "fuel_type"]

# --- Normalized-only columns ---
# Skip unit columns (metadata) and equipment (list)
_SKIP = {"engine_size_unit", "engine_power_unit", "mileage_unit", "fuel_consumption_unit", "equipment"}

N_NUM_COLS  = ["n_year", "n_engine_power", "n_engine_size", "n_cylinders",
               "n_mileage", "n_doors", "n_transmission_gears", "n_fuel_consumption"]
N_CAT_COLS  = ["n_brand", "n_model", "n_color", "n_body_type", "n_transmission_type",
               "n_transmission_technology", "n_engine_aspiration", "n_injection_type",
               "n_energy_source", "n_fuel_type", "n_propulsion_system", "n_drive_type",
               "n_submodel", "n_trim_level", "n_condition"]

# Keep only columns that actually exist in df_clean
N_NUM_COLS = [c for c in N_NUM_COLS if c in df_clean.columns]
N_CAT_COLS = [c for c in N_CAT_COLS if c in df_clean.columns]

print("Raw num :", RAW_NUM)
print("Raw cat :", RAW_CAT)
print("Norm num:", N_NUM_COLS)
print("Norm cat:", N_CAT_COLS)

Raw num : ['year', 'power_kw', 'power_ps', 'mileage_in_km']
Raw cat : ['brand', 'model', 'color', 'transmission_type', 'fuel_type']
Norm num: ['n_year', 'n_engine_power', 'n_engine_size', 'n_cylinders', 'n_mileage', 'n_doors', 'n_transmission_gears', 'n_fuel_consumption']
Norm cat: ['n_brand', 'n_model', 'n_color', 'n_body_type', 'n_transmission_type', 'n_transmission_technology', 'n_engine_aspiration', 'n_injection_type', 'n_energy_source', 'n_fuel_type', 'n_propulsion_system', 'n_drive_type', 'n_submodel', 'n_trim_level', 'n_condition']


In [6]:
from typing import Dict, List, Optional, Tuple

def impute_column(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    column: str,
    is_numeric: bool,
    min_group_size: int = 5,
    group_by: Optional[Tuple[str, str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Impute missing values in `column` using brand+model median/mode from train,
    falling back to brand-only, then overall.
    """
    if group_by is not None:
        g1, g2 = group_by
    elif column.startswith("n_"):
        g1, g2 = "n_brand", "n_model"
    else:
        g1, g2 = "brand", "model"

    imputation_values = {}

    def agg_fn(x):
        if x.count() < min_group_size:
            return None
        return x.median() if is_numeric else (x.mode()[0] if len(x.mode()) > 0 else None)

    brand_model_agg = df_train.groupby([g1, g2])[column].agg(agg_fn).dropna()
    brand_agg       = df_train.groupby(g1)[column].agg(agg_fn).dropna()

    if is_numeric:
        overall = df_train[column].median()
    else:
        overall = df_train[column].mode()[0] if not df_train[column].isna().all() else None

    for (brand, model), val in brand_model_agg.items():
        imputation_values[(brand, model)] = val
    for brand, val in brand_agg.items():
        imputation_values[(brand, None)] = val
    imputation_values[("__default__", None)] = overall

    def _fill(row):
        if not pd.isna(row[column]):
            return row[column]
        b, m = row[g1], row[g2]
        return (
            imputation_values.get((b, m))
            or imputation_values.get((b, None))
            or imputation_values[("__default__", None)]
        )

    df_train = df_train.copy()
    df_test  = df_test.copy()

    if df_train[column].isna().any():
        df_train[column] = df_train.apply(_fill, axis=1)
    if df_test[column].isna().any():
        df_test[column] = df_test.apply(_fill, axis=1)

    return df_train, df_test


def impute_all(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    num_cols: List[str],
    cat_cols: List[str],
    min_group_size: int = 5,
    group_by: Optional[Tuple[str, str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    for col in num_cols:
        df_train, df_test = impute_column(df_train, df_test, col, is_numeric=True,  min_group_size=min_group_size, group_by=group_by)
    for col in cat_cols:
        df_train, df_test = impute_column(df_train, df_test, col, is_numeric=False, min_group_size=min_group_size, group_by=group_by)
    return df_train, df_test

print("Imputation helpers defined.")

Imputation helpers defined.


In [7]:
COMBINED_NUM = RAW_NUM + N_NUM_COLS
COMBINED_CAT = RAW_CAT + N_CAT_COLS

config_raw = {
    "num_columns": RAW_NUM,
    "cat_columns": RAW_CAT,
    "target": "price",
    "validation": "holdout",
    "val_size": 0.2,
    "loss_function": "MAE",
}

config_combined = {
    "num_columns": COMBINED_NUM,
    "cat_columns": COMBINED_CAT,
    "target": "price",
    "validation": "holdout",
    "val_size": 0.2,
    "loss_function": "MAE",
}

RANDOM_STATES = list(range(10))

metrics = {
    "raw+imputed":          {"mape": [], "mdape": [], "mae": []},
    "raw+norm+imputed":     {"mape": [], "mdape": [], "mae": []},
}

for rs in RANDOM_STATES:
    df_tr_raw, df_te_raw = train_test_split(df_clean, test_size=0.2, random_state=rs)
    y_te = df_te_raw["price"].values

    # --- Raw + imputed (group by brand/model) ---
    df_tr_r, df_te_r = impute_all(
        df_tr_raw, df_te_raw,
        num_cols=RAW_NUM, cat_cols=RAW_CAT,
        group_by=("brand", "model"),
    )
    m_r, _ = train_catboost(df_tr_r, config_raw)
    yp_r = np.exp(m_r.predict(df_te_r[RAW_NUM + RAW_CAT]))
    ape_r = np.abs((yp_r - y_te) / y_te)
    metrics["raw+imputed"]["mape"].append(np.mean(ape_r))
    metrics["raw+imputed"]["mdape"].append(np.median(ape_r))
    metrics["raw+imputed"]["mae"].append(np.mean(np.abs(yp_r - y_te)))

    # --- Raw + Normalized + imputed ---
    # Impute RAW cols by (brand, model), N_ cols by (n_brand, n_model)
    df_tr_c, df_te_c = impute_all(
        df_tr_raw, df_te_raw,
        num_cols=RAW_NUM, cat_cols=RAW_CAT,
        group_by=("brand", "model"),
    )
    df_tr_c, df_te_c = impute_all(
        df_tr_c, df_te_c,
        num_cols=N_NUM_COLS, cat_cols=N_CAT_COLS,
        group_by=("n_brand", "n_model"),
    )
    m_c, _ = train_catboost(df_tr_c, config_combined)
    df_te_c_pred = df_te_c[COMBINED_NUM + COMBINED_CAT].copy()
    df_te_c_pred[COMBINED_CAT] = df_te_c_pred[COMBINED_CAT].fillna("").astype(str).replace("nan", "")
    yp_c = np.exp(m_c.predict(df_te_c_pred))
    ape_c = np.abs((yp_c - y_te) / y_te)
    metrics["raw+norm+imputed"]["mape"].append(np.mean(ape_c))
    metrics["raw+norm+imputed"]["mdape"].append(np.median(ape_c))
    metrics["raw+norm+imputed"]["mae"].append(np.mean(np.abs(yp_c - y_te)))

    print(
        f"[rs={rs}]  "
        f"raw: {metrics['raw+imputed']['mape'][-1]:.4f}/{metrics['raw+imputed']['mdape'][-1]:.4f}  "
        f"raw+norm: {metrics['raw+norm+imputed']['mape'][-1]:.4f}/{metrics['raw+norm+imputed']['mdape'][-1]:.4f}"
    )

/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[rs=0]  raw: 0.1590/0.0949  raw+norm: 0.1571/0.0934


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[rs=1]  raw: 0.1574/0.0946  raw+norm: 0.1560/0.0934


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


...

[rs=7]  raw: 0.1593/0.0949  raw+norm: 0.1584/0.0941


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[rs=8]  raw: 0.1557/0.0944  raw+norm: 0.1546/0.0933


/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/diegoquezadac/dev/vn/.venv/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[rs=9]  raw: 0.1576/0.0941  raw+norm: 0.1563/0.0933


In [8]:
print(f"\n{'='*65}")
print(f"{'Model':<18} {'Avg MAPE':>10}  {'Avg MdAPE':>10}  {'Avg MAE':>12}")
print(f"{'-'*65}")
for name, m in metrics.items():
    print(f"{name:<18} {np.mean(m['mape']):>10.4f}  {np.mean(m['mdape']):>10.4f}  {np.mean(m['mae']):>12,.0f}")
print(f"{'='*65}")


Model                Avg MAPE   Avg MdAPE       Avg MAE
-----------------------------------------------------------------
raw+imputed            0.1573      0.0945         4,021
raw+norm+imputed       0.1561      0.0935         3,959
